<a href="https://colab.research.google.com/github/fernandolievano/procesamiento-del-habla/blob/main/%5BPH%5DTP5_chatobts_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP5

**a) Creación del conjunto de datos de evaluación.**

Además del dataset original que creó, debe crear un dataset de prueba o evaluación con la misma lógica: preguntas y respuestas.

**b) Debe elegir un modelo LLM de HuggingFace y al menos dos modelos de embeddings. Justifique su elección.**

**c) Implemente una clase ChatBot usando lo elegido en b).**

Puede usar cualquier base de datos vectorial: Chroma y FAISS son las más documentadas. Recuerde que sus datos para su BD conocimiento es el dataset que Ud. planteó en el TP4.

**d) Pruebe el chatbot creado en c) con las preguntas de su dataset a)**

Usando los modelos elegidos en b), observe las respuestas generadas por el chatbot comparando al menos dos modelos de embeddings.

Justifique y determine cuál elegiría para su aplicación.

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

In [ ]:
!pip install spacy --quiet
!python -m spacy download es_core_news_md -q chromadb sentence-transformers transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 18.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import spacy
import chromadb
import torch

from chromadb.config import Settings
from chromadb.utils import embedding_functions
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from sentence_transformers import SentenceTransformer

### Utils y Setup

In [ ]:
def print_title(title):
    print('===' * 30)
    print(f'🔷 {title}')
    print('===' * 30)
    print('\n')

In [ ]:
# Carga del modelo spaCy (md)
nlp = spacy.load('es_core_news_md')

> Se eligió el modelo `es_core_news_md` porque el modelo md es más potente y nos va a permitir mejorar la calidad de las respuestas del chatbot.

## Introducción



Se eligió trabajar en un **chatbot de atención al cliente para una tienda online** debido a que este tipo de apps presentan consultas frecuentes y variadas relacionadas con envíos, pagos, devoluciones, etc.

Además, trabajar con una tienda online nos resulta útil para compara distintos enfoques de chatbot ya que los usuarios pueden realizar la misma consulta usando palabras diferentes, o expresiones similares.

## a) Creación de conjunto de datos de evaluación

Usamos el dataset original de preguntas de respuestas, y además, creamos uno de evaluación con la misma lógica (preguntas y respuestas).

### Dataset original

In [ ]:
print_title('Dataset de preguntas y respuestas originales')

data = [
    {
        "pregunta": "¿Hacen envíos a todo el país?",
        "respuesta": "Sí, realizamos envíos a todo el país mediante correo privado."
    },
    {
        "pregunta": "¿Cuánto tarda en llegar mi compra?",
        "respuesta": "El tiempo de entrega depende de tu ubicación, pero suele ser entre 3 y 7 días hábiles."
    },
    {
        "pregunta": "¿Qué métodos de pago aceptan?",
        "respuesta": "Aceptamos tarjetas de crédito, débito, transferencias y billeteras virtuales."
    },
    {
        "pregunta": "¿Puedo pagar en cuotas?",
        "respuesta": "Sí, algunas tarjetas permiten pagar en cuotas sin interés."
    },
    {
        "pregunta": "¿Cómo puedo seguir mi pedido?",
        "respuesta": "Una vez despachado el pedido, te enviaremos un código de seguimiento por correo electrónico."
    },
    {
        "pregunta": "¿Tienen retiro en sucursal?",
        "respuesta": "Sí, ofrecemos retiro en sucursal sin costo adicional."
    },
    {
        "pregunta": "¿Qué pasa si el producto llega dañado?",
        "respuesta": "Si el producto llega dañado, podés solicitar un cambio o devolución dentro de las primeras 48 horas."
    },
    {
        "pregunta": "¿Puedo devolver un producto?",
        "respuesta": "Sí, aceptamos devoluciones dentro de los 10 días posteriores a la compra."
    },
    {
        "pregunta": "¿Los productos tienen garantía?",
        "respuesta": "Sí, todos nuestros productos cuentan con garantía oficial del fabricante."
    },
    {
        "pregunta": "¿Cómo creo una cuenta?",
        "respuesta": "Podés registrarte desde el botón 'Crear cuenta' completando tus datos personales."
    },
    {
        "pregunta": "Olvidé mi contraseña, ¿qué hago?",
        "respuesta": "Podés recuperar tu contraseña desde la opción 'Olvidé mi contraseña' en la pantalla de inicio de sesión."
    },
    {
        "pregunta": "¿Hay descuentos disponibles?",
        "respuesta": "Sí, regularmente ofrecemos promociones y descuentos especiales en distintos productos."
    },
    {
        "pregunta": "¿Cómo aplico un cupón de descuento?",
        "respuesta": "El cupón se puede ingresar durante el proceso de pago antes de finalizar la compra."
    },
    {
        "pregunta": "¿Tienen stock disponible?",
        "respuesta": "El stock actualizado aparece en la página de cada producto."
    },
    {
        "pregunta": "¿Puedo cancelar una compra?",
        "respuesta": "Sí, podés cancelar tu compra antes de que el pedido sea despachado."
    },
    {
        "pregunta": "¿Qué hago si mi pedido no llegó?",
        "respuesta": "Si tu pedido no llegó en la fecha estimada, podés comunicarte con soporte para revisarlo."
    },
    {
        "pregunta": "¿Venden productos originales?",
        "respuesta": "Sí, trabajamos únicamente con productos originales y oficiales."
    },
    {
        "pregunta": "¿Cómo contacto con atención al cliente?",
        "respuesta": "Podés contactarnos por correo electrónico, teléfono o chat online."
    },
    {
        "pregunta": "¿El sitio es seguro para comprar?",
        "respuesta": "Sí, utilizamos protocolos de seguridad y cifrado para proteger tus datos."
    },
    {
        "pregunta": "¿Puedo modificar la dirección de entrega?",
        "respuesta": "Sí, podés modificar la dirección antes de que el pedido sea enviado."
    }
]

df = pd.DataFrame(data)

df.sample(5)

🔷 Dataset de preguntas y respuestas originales




,pregunta,respuesta
16,¿Venden productos originales?,"Sí, trabajamos únicamente con productos origin..."
0,¿Hacen envíos a todo el país?,"Sí, realizamos envíos a todo el país mediante ..."
18,¿El sitio es seguro para comprar?,"Sí, utilizamos protocolos de seguridad y cifra..."
2,¿Qué métodos de pago aceptan?,"Aceptamos tarjetas de crédito, débito, transfe..."
12,¿Cómo aplico un cupón de descuento?,El cupón se puede ingresar durante el proceso ...


### Dataset Evaluación

Creamos un segundo conjunto de preguntas con la misma lógica del dataset original para poder utilizarlo como conjunto de prueba.

Las preguntas si bien están redactadas de forma diferente a las originales apuntan a las mismas intenciones: envíos, pagos, seguimiento, etc.

De esta manera podemos consultar al chatbot con preguntas nuevas y comparar su respuesta con la respuesta esperada.

In [ ]:
print_title('Dataset evaluación')

data_eval = [
    {
        "id_eval": 1,
        "categoria": "envios",
        "pregunta": "¿Envían productos a cualquier provincia?",
        "respuesta_esperada": "Sí, realizamos envíos a todo el país mediante correo privado.",
        "id_conocimiento": 0
    },
    {
        "id_eval": 2,
        "categoria": "envios",
        "pregunta": "¿Cuántos días demora aproximadamente la entrega?",
        "respuesta_esperada": "El tiempo de entrega depende de tu ubicación, pero suele ser entre 3 y 7 días hábiles.",
        "id_conocimiento": 1
    },
    {
        "id_eval": 3,
        "categoria": "pagos",
        "pregunta": "¿Con qué medios puedo abonar mi pedido?",
        "respuesta_esperada": "Aceptamos tarjetas de crédito, débito, transferencias y billeteras virtuales.",
        "id_conocimiento": 2
    },
    {
        "id_eval": 4,
        "categoria": "pagos",
        "pregunta": "¿Se puede comprar en cuotas?",
        "respuesta_esperada": "Sí, algunas tarjetas permiten pagar en cuotas sin interés.",
        "id_conocimiento": 3
    },
    {
        "id_eval": 5,
        "categoria": "seguimiento",
        "pregunta": "¿Dónde veo el seguimiento de mi compra?",
        "respuesta_esperada": "Una vez despachado el pedido, te enviaremos un código de seguimiento por correo electrónico.",
        "id_conocimiento": 4
    },
    {
        "id_eval": 6,
        "categoria": "retiro",
        "pregunta": "¿Puedo retirar mi compra personalmente?",
        "respuesta_esperada": "Sí, ofrecemos retiro en sucursal sin costo adicional.",
        "id_conocimiento": 5
    },
    {
        "id_eval": 7,
        "categoria": "devoluciones",
        "pregunta": "Mi producto llegó roto, ¿qué puedo hacer?",
        "respuesta_esperada": "Si el producto llega dañado, podés solicitar un cambio o devolución dentro de las primeras 48 horas.",
        "id_conocimiento": 6
    },
    {
        "id_eval": 8,
        "categoria": "devoluciones",
        "pregunta": "¿Aceptan cambios o devoluciones después de comprar?",
        "respuesta_esperada": "Sí, aceptamos devoluciones dentro de los 10 días posteriores a la compra.",
        "id_conocimiento": 7
    },
    {
        "id_eval": 9,
        "categoria": "garantia",
        "pregunta": "¿La compra incluye garantía?",
        "respuesta_esperada": "Sí, todos nuestros productos cuentan con garantía oficial del fabricante.",
        "id_conocimiento": 8
    },
    {
        "id_eval": 10,
        "categoria": "cuenta",
        "pregunta": "¿Cómo hago para registrarme en la tienda?",
        "respuesta_esperada": "Podés registrarte desde el botón 'Crear cuenta' completando tus datos personales.",
        "id_conocimiento": 9
    },
    {
        "id_eval": 11,
        "categoria": "cuenta",
        "pregunta": "No recuerdo mi clave, ¿cómo la recupero?",
        "respuesta_esperada": "Podés recuperar tu contraseña desde la opción 'Olvidé mi contraseña' en la pantalla de inicio de sesión.",
        "id_conocimiento": 10
    },
    {
        "id_eval": 12,
        "categoria": "descuentos",
        "pregunta": "¿Tienen promociones activas?",
        "respuesta_esperada": "Sí, regularmente ofrecemos promociones y descuentos especiales en distintos productos.",
        "id_conocimiento": 11
    },
    {
        "id_eval": 13,
        "categoria": "descuentos",
        "pregunta": "¿Dónde cargo un código de descuento?",
        "respuesta_esperada": "El cupón se puede ingresar durante el proceso de pago antes de finalizar la compra.",
        "id_conocimiento": 12
    },
    {
        "id_eval": 14,
        "categoria": "stock",
        "pregunta": "¿Cómo sé si un producto está disponible?",
        "respuesta_esperada": "El stock actualizado aparece en la página de cada producto.",
        "id_conocimiento": 13
    },
    {
        "id_eval": 15,
        "categoria": "cancelaciones",
        "pregunta": "¿Puedo anular mi pedido antes del envío?",
        "respuesta_esperada": "Sí, podés cancelar tu compra antes de que el pedido sea despachado.",
        "id_conocimiento": 14
    },
    {
        "id_eval": 16,
        "categoria": "soporte",
        "pregunta": "Mi compra todavía no llegó, ¿con quién hablo?",
        "respuesta_esperada": "Si tu pedido no llegó en la fecha estimada, podés comunicarte con soporte para revisarlo.",
        "id_conocimiento": 15
    },
    {
        "id_eval": 17,
        "categoria": "productos",
        "pregunta": "¿Los artículos que venden son originales?",
        "respuesta_esperada": "Sí, trabajamos únicamente con productos originales y oficiales.",
        "id_conocimiento": 16
    },
    {
        "id_eval": 18,
        "categoria": "soporte",
        "pregunta": "¿Cómo puedo hablar con atención al cliente?",
        "respuesta_esperada": "Podés contactarnos por correo electrónico, teléfono o chat online.",
        "id_conocimiento": 17
    },
    {
        "id_eval": 19,
        "categoria": "seguridad",
        "pregunta": "¿Es confiable pagar desde la página?",
        "respuesta_esperada": "Sí, utilizamos protocolos de seguridad y cifrado para proteger tus datos.",
        "id_conocimiento": 18
    },
    {
        "id_eval": 20,
        "categoria": "envios",
        "pregunta": "Me equivoqué en la dirección, ¿puedo cambiarla?",
        "respuesta_esperada": "Sí, podés modificar la dirección antes de que el pedido sea enviado.",
        "id_conocimiento": 19
    }
]

df_eval = pd.DataFrame(data_eval)

df_eval.sample(5)

🔷 Dataset evaluación




,id_eval,categoria,pregunta,respuesta_esperada,id_conocimiento
11,12,descuentos,¿Tienen promociones activas?,"Sí, regularmente ofrecemos promociones y descu...",11
7,8,devoluciones,¿Aceptan cambios o devoluciones después de com...,"Sí, aceptamos devoluciones dentro de los 10 dí...",7
10,11,cuenta,"No recuerdo mi clave, ¿cómo la recupero?",Podés recuperar tu contraseña desde la opción ...,10
16,17,productos,¿Los artículos que venden son originales?,"Sí, trabajamos únicamente con productos origin...",16
14,15,cancelaciones,¿Puedo anular mi pedido antes del envío?,"Sí, podés cancelar tu compra antes de que el p...",14


### Validación del dataset de evaluación

Nos aseguramos que el dataset de evaluación tenga la misma cantidad de registros que el dataset original mientras que no sean exactamente iguales a las preguntas originales.

De esta manera, podemos probar el chatbot de forma más realista.

In [ ]:
print_title('Validación del dataset de evaluación')

print(f'Cantidad de preguntas originales: {len(df)}')
print(f'Cantidad de preguntas de evaluación: {len(df_eval)}')

print('Valores nulos por columna:')
print(df_eval.isnull().sum())

preguntas_originales = set(df['pregunta'].str.lower().str.strip())
preguntas_eval = set(df_eval['pregunta'].str.lower().str.strip())

preguntas_repetidas = preguntas_originales.intersection(preguntas_eval)

print(f'Preguntas repetidas exactamente: {len(preguntas_repetidas)}')

if len(preguntas_repetidas) > 0:
    print(preguntas_repetidas)
else:
    print('No hay preguntas repetidas exactamente entre el dataset original y el de evaluación.')

df_eval.sample(5)

🔷 Validación del dataset de evaluación


Cantidad de preguntas originales: 20
Cantidad de preguntas de evaluación: 20
Valores nulos por columna:
id_eval               0
categoria             0
pregunta              0
respuesta_esperada    0
id_conocimiento       0
dtype: int64
Preguntas repetidas exactamente: 0
No hay preguntas repetidas exactamente entre el dataset original y el de evaluación.


,id_eval,categoria,pregunta,respuesta_esperada,id_conocimiento
19,20,envios,"Me equivoqué en la dirección, ¿puedo cambiarla?","Sí, podés modificar la dirección antes de que ...",19
17,18,soporte,¿Cómo puedo hablar con atención al cliente?,"Podés contactarnos por correo electrónico, tel...",17
9,10,cuenta,¿Cómo hago para registrarme en la tienda?,Podés registrarte desde el botón 'Crear cuenta...,9
14,15,cancelaciones,¿Puedo anular mi pedido antes del envío?,"Sí, podés cancelar tu compra antes de que el p...",14
5,6,retiro,¿Puedo retirar mi compra personalmente?,"Sí, ofrecemos retiro en sucursal sin costo adi...",5


### Justificación del conjunto de evaluación

El dataset de evaluación mantiene la estructura e intención de llas preguntas y respuestas originales pero se intentó usar preguntas alternativas para intentar representar consultas reales de usuarios.

Se agregó una columna `categoría` para poder identificar el tipo de consulta y una columna `id_conocimiento` para relacionar cada pregunta de evaluación con su "registro esperado" del dataset original.

In [ ]:
print_title('Base de conocimiento')

df_conocimiento = df.copy().reset_index()
df_conocimiento = df_conocimiento.rename(columns={'index': 'id'})
df_conocimiento["doc"] = (
    "pregunta: "+ df_conocimiento["pregunta"] + "\n" +
    "respuesta: "+ df_conocimiento["respuesta"]
)
df_conocimiento.sample(5)

🔷 Base de conocimiento




,id,pregunta,respuesta,doc
10,10,"Olvidé mi contraseña, ¿qué hago?",Podés recuperar tu contraseña desde la opción ...,"pregunta: Olvidé mi contraseña, ¿qué hago?\nre..."
0,0,¿Hacen envíos a todo el país?,"Sí, realizamos envíos a todo el país mediante ...",pregunta: ¿Hacen envíos a todo el país?\nrespu...
14,14,¿Puedo cancelar una compra?,"Sí, podés cancelar tu compra antes de que el p...",pregunta: ¿Puedo cancelar una compra?\nrespues...
3,3,¿Puedo pagar en cuotas?,"Sí, algunas tarjetas permiten pagar en cuotas ...",pregunta: ¿Puedo pagar en cuotas?\nrespuesta: ...
19,19,¿Puedo modificar la dirección de entrega?,"Sí, podés modificar la dirección antes de que ...",pregunta: ¿Puedo modificar la dirección de ent...


In [ ]:
print_title('Colección en ChromaDB')

embbeding_model_1  = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedding_function_1 = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name=embbeding_model_1
)
client = chromadb.Client()

collection_1 = client.get_or_create_collection(
    name="tienda_online_multilingual",
    embedding_function=embedding_function_1
)

🔷 Colección en ChromaDB




In [ ]:
print_title('Carga de documentos y mini test')


ids = df_conocimiento["id"].astype(str).tolist()

documents = df_conocimiento["doc"].tolist()

metadatas = df_conocimiento[["pregunta", "respuesta"]].to_dict(
    orient="records"
)

collection_1.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)


resultado = collection_1.query(
    query_texts=["¿Cuánto tarda en llegar mi pedido?"],
    n_results=3
)

resultado

🔷 Carga de documentos y mini test




{'ids': [['1', '15', '19']],
 'embeddings': None,
 'documents': [['pregunta: ¿Cuánto tarda en llegar mi compra?\nrespuesta: El tiempo de entrega depende de tu ubicación, pero suele ser entre 3 y 7 días hábiles.',
   'pregunta: ¿Qué hago si mi pedido no llegó?\nrespuesta: Si tu pedido no llegó en la fecha estimada, podés comunicarte con soporte para revisarlo.',
   'pregunta: ¿Puedo modificar la dirección de entrega?\nrespuesta: Sí, podés modificar la dirección antes de que el pedido sea enviado.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'respuesta': 'El tiempo de entrega depende de tu ubicación, pero suele ser entre 3 y 7 días hábiles.',
    'pregunta': '¿Cuánto tarda en llegar mi compra?'},
   {'pregunta': '¿Qué hago si mi pedido no llegó?',
    'respuesta': 'Si tu pedido no llegó en la fecha estimada, podés comunicarte con soporte para revisarlo.'},
   {'pregunta': '¿Puedo modificar la dirección de entrega?',
    'respues

## b) Elección de modelos LLM HuggingFace, embeddings y justificación.

Para seleccionar el modelo LLM de HuggingFace y los dos modelos de embeddings el criterio fue:
- Que los modelos puedan ejecutarse en Colab sin requerir demasiados recursos.
- Que sean adecuados para una aplicación como la nuestra que se basa en recuperación de información sobre una base de conocimiento.

In [ ]:
print_title('Modelos seleccionados')

llm_model_name = "google/flan-t5-small"

embedding_models = {
    "miniLM_multilingual": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "distiluse_multilingual": "sentence-transformers/distiluse-base-multilingual-cased-v2"
}

print("Modelo LLM seleccionado:")
print(f"- {llm_model_name}")

print("\nModelos de embeddings seleccionados:")
for nombre, modelo in embedding_models.items():
    print(f"- {nombre}: {modelo}")

🔷 Modelos seleccionados


Modelo LLM seleccionado:
- google/flan-t5-small

Modelos de embeddings seleccionados:
- miniLM_multilingual: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
- distiluse_multilingual: sentence-transformers/distiluse-base-multilingual-cased-v2


### Selección de modelos





Se eligió `google/flan-t5-small` como modelo LLM para el chatbot, y se seleccionó porque:
- Es más liviano comparado con otros modelos generativos similares
- Puede utilizarse en Colab requiriendo menos memoria
- Es adecuado para tareas de pregunta-respuesta y generación mínima de texto

> En general, este modelo resulta suficiente para este trabajo, ya que no necesitamos generar respuestas extensas, sino responder a las preguntas a partir de una base de conocimiento existente.


Como primer modelo de embeddings se seleccionó  `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` por los siguientes motivos:
- Es un modelo multilingüe, nos va a permitir trabajar con nuestras preguntas en español
- Está preparado para trabajar con similitud semántica entre frases
- Genera embeddings de 384 dimensiones


Como segundo modelo de embeddings se eligió  `sentence-transformers/distiluse-base-multilingual-cased-v2` teniendo en cuenta que:
- Nos permite trabajar en español al ser multilingual
- Pertenece a la familia Sentece Transformers
- Nos va a permitir comparar los resultados de dos modelos multilingües

## c) Implementación de Chatbot

In [ ]:
class ChatBot:
    def __init__(
        self,
        df_conocimiento,
        embedding_model_name,
        llm_model_name,
        collection_name="chatbot_collection",
        pregunta_col="pregunta",
        respuesta_col="respuesta"
    ):
        self.df_conocimiento = df_conocimiento.copy()
        self.embedding_model_name = embedding_model_name
        self.llm_model_name = llm_model_name
        self.collection_name = collection_name
        self.pregunta_col = pregunta_col
        self.respuesta_col = respuesta_col

        self._validar_dataset()
        self._cargar_modelo_embeddings()
        self._cargar_modelo_llm()
        self._crear_base_vectorial()

    def _validar_dataset(self):
        columnas_requeridas = [self.pregunta_col, self.respuesta_col]

        for columna in columnas_requeridas:
            if columna not in self.df_conocimiento.columns:
                raise ValueError(f"El dataset debe tener una columna llamada '{columna}'")

        self.df_conocimiento = self.df_conocimiento.dropna(
            subset=[self.pregunta_col, self.respuesta_col]
        ).reset_index(drop=True)

    def _cargar_modelo_embeddings(self):
        print(f"Cargando modelo de embeddings: {self.embedding_model_name}")
        self.embedding_model = SentenceTransformer(self.embedding_model_name)

    def _cargar_modelo_llm(self):
        print(f"Cargando modelo LLM: {self.llm_model_name}")

        self.tokenizer = AutoTokenizer.from_pretrained(self.llm_model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(self.llm_model_name)

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = self.model.to(self.device)

    def _preparar_documentos(self):
        documentos = []
        metadatos = []
        ids = []

        for index, row in self.df_conocimiento.iterrows():
            pregunta = str(row[self.pregunta_col])
            respuesta = str(row[self.respuesta_col])

            documento = f"Pregunta: {pregunta}\nRespuesta: {respuesta}"

            metadata = {
                "pregunta": pregunta,
                "respuesta": respuesta
            }

            if "categoria" in self.df_conocimiento.columns:
                metadata["categoria"] = str(row["categoria"])

            documentos.append(documento)
            metadatos.append(metadata)
            ids.append(f"doc_{index}")

        return documentos, metadatos, ids

    def _crear_base_vectorial(self):
        print("Creando base vectorial con Chroma...")

        self.client = chromadb.Client()

        try:
            self.client.delete_collection(name=self.collection_name)
        except Exception:
            pass

        self.collection = self.client.create_collection(
            name=self.collection_name
        )

        documentos, metadatos, ids = self._preparar_documentos()

        embeddings = self.embedding_model.encode(
            documentos,
            convert_to_numpy=True
        ).tolist()

        self.collection.add(
            documents=documentos,
            metadatas=metadatos,
            ids=ids,
            embeddings=embeddings
        )

        print(f"Documentos cargados en Chroma: {len(documentos)}")

    def buscar_contexto(self, pregunta, n_resultados=3):
        pregunta_embedding = self.embedding_model.encode(
            [pregunta],
            convert_to_numpy=True
        ).tolist()

        resultados = self.collection.query(
            query_embeddings=pregunta_embedding,
            n_results=n_resultados
        )

        documentos = resultados["documents"][0]
        metadatos = resultados["metadatas"][0]
        distancias = resultados["distances"][0]

        return documentos, metadatos, distancias

    def generar_prompt(self, pregunta, documentos):
        contexto = "\n\n".join(documentos)

        prompt = f"""
                    Respondé la pregunta del usuario usando únicamente la información del contexto.

                    Contexto:
                    {contexto}

                    Pregunta del usuario:
                    {pregunta}

                    Respuesta:
                    """

        return prompt

    def responder(self, pregunta, n_resultados=3, mostrar_contexto=False):
        documentos, metadatos, distancias = self.buscar_contexto(
            pregunta,
            n_resultados=n_resultados
        )

        prompt = self.generar_prompt(pregunta, documentos)

        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(self.device)

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False
        )

        respuesta_generada = self.tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )

        resultado = {
            "pregunta": pregunta,
            "respuesta": respuesta_generada,
            "documentos_recuperados": documentos,
            "metadatos": metadatos,
            "distancias": distancias
        }

        if mostrar_contexto:
            print("Pregunta:")
            print(pregunta)

            print("\nRespuesta generada:")
            print(respuesta_generada)

            print("\nDocumentos recuperados:")
            for i, documento in enumerate(documentos):
                print(f"\nDocumento {i + 1}:")
                print(documento)
                print(f"Distancia: {distancias[i]}")

        return resultado

> La clase `ChatBot` recibe como entrada el dataset de conocimiento, un modelo de embeddings y un modelo LLM.
>
> Al inicializarse, la clase valida que el dataset tenga las columnas necesarias, luego carga el modelo de embeddings, carga el modelo generativo y crea una colección en Chroma.
>
> Después, cada fila del dataset se convierte en un documento compuesto por una pregunta y una respuesta. Luego, cada documento se transforma en un vector mediante el modelo de embeddings seleccionado y se almacena en Chroma.
>
> Cuando el usuario realiza una pregunta  el chatbot genera el embedding de esa consulta y busca en Chroma los documentos más similares. Luego esos documentos se utilizan como contexto para que el modelo LLM genere una respuesta adecuada.
>
> Esta implementación permite cambiar fácilmente el modelo de embeddings lo cual será útil para comparar los resultados en la consigna siguiente.

### Chatbot con primer modelo de embeddings (miniLM-multilingual)

In [ ]:
print_title("ChatBot con paraphrase-multilingual-MiniLM-L12-v2")

chatbot_minilm = ChatBot(
    df_conocimiento=df_conocimiento,
    embedding_model_name=embedding_models["miniLM_multilingual"],
    llm_model_name=llm_model_name,
    collection_name="chatbot_minilm_multilingual"
)

🔷 ChatBot con paraphrase-multilingual-MiniLM-L12-v2


Cargando modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Cargando modelo LLM: google/flan-t5-small


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Creando base vectorial con Chroma...
Documentos cargados en Chroma: 20


In [ ]:
print_title('Test con pregunta')

respuesta = chatbot_minilm.responder(
    "¿Cuánto tarda en llegar mi pedido?",
    mostrar_contexto=True
)

🔷 Test con pregunta


Pregunta:
¿Cuánto tarda en llegar mi pedido?

Respuesta generada:
El tiempo de entrega depende de tu ubicación, pero suele ser entre 3 y 7 das hábiles.

Documentos recuperados:

Documento 1:
Pregunta: ¿Cuánto tarda en llegar mi compra?
Respuesta: El tiempo de entrega depende de tu ubicación, pero suele ser entre 3 y 7 días hábiles.
Distancia: 7.683182716369629

Documento 2:
Pregunta: ¿Qué hago si mi pedido no llegó?
Respuesta: Si tu pedido no llegó en la fecha estimada, podés comunicarte con soporte para revisarlo.
Distancia: 11.610233306884766

Documento 3:
Pregunta: ¿Cómo puedo seguir mi pedido?
Respuesta: Una vez despachado el pedido, te enviaremos un código de seguimiento por correo electrónico.
Distancia: 15.174009323120117


### Chatbot con segundo modelo de embeddings (distiluse_multilingual)

In [ ]:
print_title("ChatBot con distiluse-base-multilingual-cased-v2")

chatbot_minilm_v2 = ChatBot(
    df_conocimiento=df_conocimiento,
    embedding_model_name=embedding_models["distiluse_multilingual"],
    llm_model_name=llm_model_name,
    collection_name="chatbot_distiluse_multilingual"
)

🔷 ChatBot con distiluse-base-multilingual-cased-v2


Cargando modelo de embeddings: sentence-transformers/distiluse-base-multilingual-cased-v2


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Cargando modelo LLM: google/flan-t5-small


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Creando base vectorial con Chroma...
Documentos cargados en Chroma: 20


In [ ]:
print_title('Test con pregunta')

respuesta = chatbot_minilm_v2.responder(
    "¿Cuánto tarda en llegar mi pedido?",
    mostrar_contexto=True
)

🔷 Test con pregunta


Pregunta:
¿Cuánto tarda en llegar mi pedido?

Respuesta generada:
El tiempo de entrega depende de tu ubicación, pero suele ser entre 3 y 7 das hábiles.

Documentos recuperados:

Documento 1:
Pregunta: ¿Cuánto tarda en llegar mi compra?
Respuesta: El tiempo de entrega depende de tu ubicación, pero suele ser entre 3 y 7 días hábiles.
Distancia: 0.6216656565666199

Documento 2:
Pregunta: ¿Qué hago si mi pedido no llegó?
Respuesta: Si tu pedido no llegó en la fecha estimada, podés comunicarte con soporte para revisarlo.
Distancia: 0.8095858693122864

Documento 3:
Pregunta: ¿Cómo puedo seguir mi pedido?
Respuesta: Una vez despachado el pedido, te enviaremos un código de seguimiento por correo electrónico.
Distancia: 0.8528756499290466


### Test de bots con input

In [ ]:
print_title('Test miniLM-multilingual')

pregunta = input('Hola, ¿Cómo podemos ayudarte?')

while pregunta.lower() != 'salir':
    respuesta = chatbot_minilm.responder(pregunta, mostrar_contexto=True)
    print(respuesta['respuesta'])
    print(f'\n {'===' * 30} \n')
    pregunta = input()

🔷 Test miniLM-multilingual


Hola, ¿Cómo podemos ayudarte?Quiero hacer una devolución
Pregunta:
Quiero hacer una devolución

Respuesta generada:
S, acceptamos devoluciones dentro de los 10 das posteriores a la compra.

Documentos recuperados:

Documento 1:
Pregunta: ¿Puedo devolver un producto?
Respuesta: Sí, aceptamos devoluciones dentro de los 10 días posteriores a la compra.
Distancia: 12.345279693603516

Documento 2:
Pregunta: ¿Qué hago si mi pedido no llegó?
Respuesta: Si tu pedido no llegó en la fecha estimada, podés comunicarte con soporte para revisarlo.
Distancia: 13.328271865844727

Documento 3:
Pregunta: ¿Cómo aplico un cupón de descuento?
Respuesta: El cupón se puede ingresar durante el proceso de pago antes de finalizar la compra.
Distancia: 13.427556037902832
S, acceptamos devoluciones dentro de los 10 das posteriores a la compra.


Me gustaría saber si puedo pagar con tarjeta de crédito
Pregunta:
Me gustaría saber si puedo pagar con tarjeta de crédito

Respuesta generada

In [ ]:

print_title('Test distiluse_multilingual')

pregunta = input('Hola, ¿Cómo podemos ayudarte?')

while pregunta.lower() != 'salir':
    respuesta = chatbot_minilm_v2.responder(pregunta, mostrar_contexto=True)
    print(respuesta['respuesta'])
    print(f'\n {'===' * 30} \n')
    pregunta = input()

🔷 Test distiluse_multilingual


Hola, ¿Cómo podemos ayudarte?Hola, puedo hacer una devolución?
Pregunta:
Hola, puedo hacer una devolución?

Respuesta generada:
Por qué aceptamos devoluciones dentro de los 10 das posteriores a la compra.

Documentos recuperados:

Documento 1:
Pregunta: Olvidé mi contraseña, ¿qué hago?
Respuesta: Podés recuperar tu contraseña desde la opción 'Olvidé mi contraseña' en la pantalla de inicio de sesión.
Distancia: 0.8110450506210327

Documento 2:
Pregunta: ¿Puedo devolver un producto?
Respuesta: Sí, aceptamos devoluciones dentro de los 10 días posteriores a la compra.
Distancia: 0.8457050323486328

Documento 3:
Pregunta: ¿Puedo modificar la dirección de entrega?
Respuesta: Sí, podés modificar la dirección antes de que el pedido sea enviado.
Distancia: 0.8583623766899109
Por qué aceptamos devoluciones dentro de los 10 das posteriores a la compra.


¿y puedo pagar con tarjeta de crédito?
Pregunta:
¿y puedo pagar con tarjeta de crédito?

Respuesta generada:
S, 

### Comparación

Después de comparar los rasultados de ambos chatbots podemos observar que:

- **chatbot_minilm**: Este modelo recupera documentos con distancias más altas, lo que significa que la pregunta y el documento recuperado no están tan cerca entre sí según la representación que hace el modelo. Aunque muchas veces acierta el documento correcto estas distancias más altas pueden hacer que el modelo tenga más dificultad para identificar con precisión el contexto en preguntas más complejas.
- **chatbot_distiluse_multilingual**: Por contraparte, este modelo muestra distancias más bajas cercanas a cero en los documentos recuperados, lo que indica que logra encontrar documentos más cercanos a la pregunta del usuario. En otras palabras, parece representar mejor la relación entre la consulta y la información disponible para poder respoder de manera más precisa.

# Referencias

### Uso de IA:
- [Generación de preguntas y respuestas para el dataset](https://chatgpt.com/share/6a0bed2e-7eb4-83e9-89f8-5c4d8921726b)


